# Learning Methodology: How to Use leaps Effectively

> **Repository:** leaps — Learning Environment for Any Progressive Subject  
> **Type:** Cross-topic methodology notebook  
> **Prerequisites:** None — this notebook is for every learner  

---

This notebook covers the cognitive science behind effective learning and shows concretely how leaps is designed around these principles. Understanding *why* the system is structured the way it is will make you a more effective user of it.

**What you will learn:**
- Why we forget — and what the data says about memory decay
- How spaced repetition fights the forgetting curve
- Why active recall outperforms passive review by 2–4×
- How interleaving practice improves long-term retention
- How to use every part of leaps to maximize each technique

**Time to complete:** ~45 minutes (reading + running cells)

## 1. The Science of Learning

### 1.1 The Forgetting Curve (Ebbinghaus, 1885)

Hermann Ebbinghaus was the first to measure memory decay empirically. He memorized lists of nonsense syllables, then tested himself at intervals to measure what percentage he retained. The result: **memory decays exponentially with time** — most rapidly in the first hours after learning, then more slowly.

The simplified model:

$$R(t) = e^{-t/S}$$

where:
- $R(t)$ = retention (fraction of material retained) at time $t$
- $t$ = time since learning (in days)
- $S$ = stability of the memory (increases with each successful review)

**Key finding:** Without review, you forget ~50–70% of new material within 24 hours and ~80–90% within a week.

### 1.2 The Spacing Effect

The spacing effect (also Ebbinghaus) shows that **spreading reviews over time is dramatically more effective than massed practice (cramming)**. Reviewing material after a delay — just before you would forget it — strengthens the memory more than reviewing it immediately.

### 1.3 The Testing Effect (Retrieval Practice)

Roediger & Karpicke (2006): students who tested themselves on material remembered significantly more after one week than students who spent the same time re-reading. Retrieval practice does not just *measure* learning — it *produces* learning.

### 1.4 Interleaving

Interleaving (mixing different topics or problem types in a single study session) feels harder than blocked practice (drilling one thing at a time) but produces better long-term retention and transfer. The difficulty is the point — it forces your brain to discriminate between concepts rather than just pattern-matching to the most recent example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Ebbinghaus forgetting curve ──────────────────────────────────────────────
# R(t) = exp(-t / S)
# S (stability) increases with each review.
# We model multiple review sessions by resetting the curve at each review point
# with an increased stability value.

t = np.linspace(0, 30, 1000)  # days

# Stability values: S increases with each review (approximation)
INITIAL_STABILITY = 1.5   # days — initial memory fades quickly
STABILITY_MULTIPLIER = 2.5  # each review multiplies stability

def retention_curve(t_range, t_start, stability):
    """Retention at each point in t_range, measured from t_start."""
    return 100 * np.exp(-(t_range - t_start) / stability)

# Without any review
r_no_review = retention_curve(t, 0, INITIAL_STABILITY)

# With spaced reviews (review at the dip threshold: ~70% retention)
REVIEW_THRESHOLD = 70  # review when retention drops to 70%

reviews = []
stability = INITIAL_STABILITY
r_spaced = np.zeros_like(t)

# Build the spaced-review retention curve piecewise
start = 0.0
for i, time_point in enumerate(t):
    segment_r = 100 * np.exp(-(time_point - start) / stability)
    r_spaced[i] = segment_r
    # If we drop to threshold AND we haven't reviewed in the last 0.1 days
    if segment_r <= REVIEW_THRESHOLD and (not reviews or time_point - reviews[-1] > 0.1):
        reviews.append(time_point)
        stability *= STABILITY_MULTIPLIER
        start = time_point
        r_spaced[i] = 100  # retention resets to 100% at review

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("The Forgetting Curve & Spaced Repetition", fontsize=14, fontweight="bold")

# Left: forgetting curve without review
ax1 = axes[0]
ax1.plot(t, r_no_review, color="#e74c3c", linewidth=2.5, label="No review")
ax1.axhline(y=REVIEW_THRESHOLD, color="#95a5a6", linestyle="--", linewidth=1, label=f"{REVIEW_THRESHOLD}% threshold")
ax1.fill_between(t, r_no_review, 0, alpha=0.15, color="#e74c3c")
ax1.set_title("Without Review", fontsize=11)
ax1.set_xlabel("Days since learning")
ax1.set_ylabel("Retention (%)")
ax1.set_ylim(0, 105)
ax1.set_xlim(0, 30)
ax1.yaxis.set_major_formatter(mticker.PercentFormatter())
ax1.legend()
ax1.grid(True, alpha=0.3)
# Annotate the drop
ax1.annotate(
    f"~{r_no_review[np.searchsorted(t, 1)]:.0f}% after 1 day",
    xy=(1, r_no_review[np.searchsorted(t, 1)]),
    xytext=(5, 60),
    arrowprops=dict(arrowstyle="->", color="#333"),
    fontsize=9
)

# Right: spaced repetition
ax2 = axes[1]
ax2.plot(t, r_no_review, color="#e74c3c", linewidth=1.5, linestyle="--", alpha=0.4, label="No review (reference)")
ax2.plot(t, r_spaced, color="#2ecc71", linewidth=2.5, label="Spaced repetition")
ax2.axhline(y=REVIEW_THRESHOLD, color="#95a5a6", linestyle="--", linewidth=1, label=f"{REVIEW_THRESHOLD}% threshold")
for i, rv in enumerate(reviews[:6]):  # annotate first 6 reviews
    ax2.axvline(x=rv, color="#3498db", linewidth=1, alpha=0.6)
    ax2.text(rv + 0.2, 95, f"R{i+1}", fontsize=8, color="#3498db")
ax2.fill_between(t, r_spaced, REVIEW_THRESHOLD, where=(r_spaced >= REVIEW_THRESHOLD), alpha=0.1, color="#2ecc71")
ax2.set_title("With Spaced Repetition", fontsize=11)
ax2.set_xlabel("Days since learning")
ax2.set_ylabel("Retention (%)")
ax2.set_ylim(0, 105)
ax2.set_xlim(0, 30)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../assets/images/forgetting-curve-plot.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Reviews triggered at days: {[f'{r:.1f}' for r in reviews[:8]]}")

## 2. How Spaced Repetition Combats Forgetting

The plot above shows the key insight: **each review resets retention to 100% AND increases the stability of the memory**, meaning the next review can be scheduled further in the future. Over time, review intervals grow:

| Review | Interval (days) | Cumulative days |
|--------|----------------|------------------|
| 1st    | ~1.5            | 1.5              |
| 2nd    | ~3.75           | 5.25             |
| 3rd    | ~9.4            | 14.7             |
| 4th    | ~23.4           | 38.1             |
| 5th    | ~58.6           | 96.7             |

This is the core of **Spaced Repetition Systems (SRS)** like Anki, SuperMemo, and the review system in leaps. Instead of reviewing everything every day (wasteful) or never reviewing (forgetting), you review each item just before you would forget it — the most efficient possible schedule.

### The SM-2 Algorithm

SuperMemo 2 (SM-2) is the most widely used SRS algorithm. It calculates the next review interval based on:
- **Easiness factor (EF):** A float starting at 2.5, modified by how easy each recall was.
- **Repetition number (n):** How many times the item has been successfully reviewed.
- **Quality of recall (q):** Self-reported score 0–5 after each review.

The interval formula:
- If n = 1: interval = 1 day
- If n = 2: interval = 6 days
- If n > 2: interval = previous_interval × EF
- EF update: `EF = EF + (0.1 - (5-q) * (0.08 + (5-q) * 0.02))`
- If EF < 1.3, clamp to 1.3

In [ ]:
from dataclasses import dataclass, field
from datetime import date, timedelta
from typing import List

@dataclass
class SRSCard:
    """A single flashcard item in the SM-2 spaced repetition system."""
    name: str
    easiness_factor: float = 2.5
    interval: int = 0          # days until next review
    repetitions: int = 0       # successful review count
    next_review: date = field(default_factory=date.today)
    history: List[dict] = field(default_factory=list)

    def review(self, quality: int) -> "SRSCard":
        """
        Update card after a review.

        quality: 0-5
            0 — complete blackout
            1 — incorrect, but remembered when shown
            2 — incorrect, easy recall after seeing answer
            3 — correct, but with significant difficulty
            4 — correct, with some hesitation
            5 — perfect recall, no hesitation
        """
        assert 0 <= quality <= 5, "Quality must be 0-5"

        # SM-2 interval calculation
        if quality < 3:  # failed — reset
            self.repetitions = 0
            self.interval = 1
        else:  # successful
            if self.repetitions == 0:
                self.interval = 1
            elif self.repetitions == 1:
                self.interval = 6
            else:
                self.interval = round(self.interval * self.easiness_factor)
            self.repetitions += 1

        # Update easiness factor
        self.easiness_factor = max(
            1.3,
            self.easiness_factor + (0.1 - (5 - quality) * (0.08 + (5 - quality) * 0.02))
        )

        self.next_review = date.today() + timedelta(days=self.interval)
        self.history.append({
            "date": str(date.today()),
            "quality": quality,
            "interval": self.interval,
            "ef": round(self.easiness_factor, 3)
        })
        return self

    def __str__(self):
        return (
            f"Card: {self.name}\n"
            f"  Next review: {self.next_review} (in {self.interval} day(s))\n"
            f"  Easiness: {self.easiness_factor:.2f}  Repetitions: {self.repetitions}"
        )


# ── Simulate learning a concept over multiple sessions ─────────────────────────
card = SRSCard("Python decorators")

# Simulate study sessions: quality scores over time
# (3 = hard but correct, 4 = good, 5 = easy)
session_qualities = [3, 4, 3, 5, 4, 5, 5]

print("=" * 50)
print(f"Simulating SRS for: '{card.name}'")
print("=" * 50)

for session_num, quality in enumerate(session_qualities, 1):
    card.review(quality)
    quality_labels = {0: "blackout", 1: "wrong", 2: "wrong (easy)",
                      3: "hard", 4: "good", 5: "easy"}
    print(f"Session {session_num} — quality: {quality} ({quality_labels[quality]})")
    print(f"  Next review in {card.interval} day(s) | EF: {card.easiness_factor:.3f}")

print()
print("Review history:")
for h in card.history:
    print(f"  {h}")

# ── Plot interval growth ──────────────────────────────────────────────────────
intervals = [h["interval"] for h in card.history]
efs = [h["ef"] for h in card.history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("SM-2 Algorithm: Interval and Easiness Growth", fontweight="bold")

ax1.bar(range(1, len(intervals)+1), intervals, color="#3498db", edgecolor="white")
ax1.set_xlabel("Review Session")
ax1.set_ylabel("Days until next review")
ax1.set_title("Review Interval Growth")
ax1.grid(True, axis="y", alpha=0.3)
for i, v in enumerate(intervals):
    ax1.text(i+1, v + 0.3, str(v), ha="center", fontsize=9)

ax2.plot(range(1, len(efs)+1), efs, marker="o", color="#e67e22", linewidth=2)
ax2.axhline(y=2.5, color="#95a5a6", linestyle="--", label="Initial EF (2.5)")
ax2.axhline(y=1.3, color="#e74c3c", linestyle=":", label="Minimum EF (1.3)")
ax2.set_xlabel("Review Session")
ax2.set_ylabel("Easiness Factor")
ax2.set_title("Easiness Factor Evolution")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Active Recall vs. Passive Review

### The Study Trap

The most common study method — re-reading notes — is one of the least effective. It feels productive because the material becomes familiar (fluency illusion), but familiarity is not the same as retrieval ability. On a test, you need to retrieve information from scratch, not recognize it on a page.

**Passive review** (re-reading, highlighting, re-watching) creates familiarity without building retrieval strength.

**Active recall** (testing yourself, writing from memory, explaining to others) forces retrieval, which strengthens the memory trace each time it's exercised.

### The Evidence

Roediger & Karpicke (2006) — landmark experiment:
- Group A: Read a passage 4 times (massed re-reading)
- Group B: Read once, test 3 times (retrieval practice)

| Group | After 5 minutes | After 1 week |
|-------|----------------|---------------|
| Re-read 4×  | ~81%  | ~40%  |
| Test 3×     | ~75%  | ~61%  |

After one week, the testing group retained **53% more** despite scoring lower immediately after study.

### Techniques Ranked by Effectiveness

| Technique | Effectiveness | Effort |
|-----------|--------------|--------|
| Practice testing (active recall) | **High** | High |
| Distributed practice (spaced repetition) | **High** | Medium |
| Elaborative interrogation ("why?") | Medium | Medium |
| Self-explanation | Medium | Medium |
| Interleaved practice | Medium | High (feels hard) |
| Highlighting / underlining | Low | Low |
| Re-reading | Low | Low |
| Summarizing | Low-medium | Medium |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Simulate retention over 30 days: active vs. passive study ─────────────────
# We model retention as a function of time and study method,
# based on the empirical findings summarized above.

days = np.arange(0, 31)

# ── Active recall: higher initial encoding + stronger review benefit ───────────
# Simulate 3 spaced review sessions at days 1, 6, and 20
# Each review boosts retention and increases stability

def simulate_retention_active(days, review_days, base_stability=1.5, boost_factor=2.8):
    """Model active recall with spaced reviews."""
    retention = np.zeros(len(days))
    stability = base_stability
    last_review = 0
    for i, d in enumerate(days):
        if d in review_days:
            stability *= boost_factor
            last_review = d
        retention[i] = 100 * np.exp(-(d - last_review) / stability)
    return retention


def simulate_retention_passive(days, reread_days, base_stability=1.0, boost_factor=1.3):
    """Model passive re-reading: small stability boost each time."""
    retention = np.zeros(len(days))
    stability = base_stability
    last_review = 0
    for i, d in enumerate(days):
        if d in reread_days:
            stability *= boost_factor
            last_review = d
        retention[i] = 100 * np.exp(-(d - last_review) / stability)
    return retention


# Active: 3 sessions on days 1, 6, 20
active_reviews = {1, 6, 20}
active_retention = simulate_retention_active(days, active_reviews)

# Passive: same number of sessions, same days — but smaller effect
passive_retention = simulate_retention_passive(days, active_reviews)

# Heavy passive: 6 re-reads on days 1, 3, 6, 10, 15, 22
heavy_passive_reviews = {1, 3, 6, 10, 15, 22}
heavy_passive_retention = simulate_retention_passive(days, heavy_passive_reviews)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(days, active_retention, color="#2ecc71", linewidth=2.5, label="Active recall (3 sessions)")
ax.plot(days, passive_retention, color="#e74c3c", linewidth=2, linestyle="--",
        label="Passive re-read (3 sessions, same days)")
ax.plot(days, heavy_passive_retention, color="#f39c12", linewidth=2, linestyle=":",
        label="Heavy passive re-read (6 sessions)")

# Mark review points for active
for rd in sorted(active_reviews):
    idx = np.searchsorted(days, rd)
    ax.axvline(x=rd, color="#2ecc71", alpha=0.3, linewidth=1)
    ax.scatter([rd], [active_retention[idx]], color="#2ecc71", zorder=5, s=60)

ax.set_xlabel("Days since initial study", fontsize=11)
ax.set_ylabel("Estimated Retention (%)", fontsize=11)
ax.set_title("Active Recall vs. Passive Review: Simulated Retention Over 30 Days",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, 105)
ax.set_xlim(0, 30)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}%"))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Annotate final values
for label, data, color in [
    ("Active: {:.0f}%", active_retention, "#27ae60"),
    ("Passive (3×): {:.0f}%", passive_retention, "#c0392b"),
    ("Passive (6×): {:.0f}%", heavy_passive_retention, "#e67e22")
]:
    ax.text(30.5, data[-1], label.format(data[-1]), va="center", color=color, fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nDay 30 retention estimates:")
print(f"  Active recall (3 sessions):    {active_retention[-1]:.1f}%")
print(f"  Passive re-read (3 sessions):  {passive_retention[-1]:.1f}%")
print(f"  Heavy passive (6 sessions):    {heavy_passive_retention[-1]:.1f}%")
print(f"\nActive vs. passive advantage: {active_retention[-1] / max(passive_retention[-1], 0.1):.1f}×")

## 4. Using leaps for Each Technique

Every structural element of leaps is designed to support one or more evidence-based learning techniques.

### Spaced Repetition in leaps

leaps supports spaced repetition through:

| Feature | How to use it |
|---------|---------------|
| `next_review` frontmatter field | Set in each module after reviewing it. Obsidian Dataview Query 9 surfaces due items. |
| `grades.md` (append-only) | Records test scores over time — your track record shows which topics need more frequent review. |
| Review sessions in modules | Each module has a `## Review` section with summary points and self-check questions. |
| Obsidian Calendar plugin | Visualize which topics you've reviewed each day over time. |

**Recommended workflow:**
1. Complete a module. Set `next_review: <date + 1 day>` in frontmatter.
2. Each day, open the Dataview dashboard (Query 9) to see due items.
3. Review due items from memory before opening the module. Score yourself 1–5.
4. Update `next_review` and `review_interval` accordingly.

### Active Recall in leaps

| Feature | How to use it |
|---------|---------------|
| `questions.md` | After reading a module, close it and write questions from memory. Then check answers. |
| `exercises/` | Coding exercises are retrieval practice — you must produce code, not recognize it. |
| Module tests | Each module has a `## Test Yourself` section — answer questions before revealing answers. |
| Teaching others | Write a short explanation in your own words at the bottom of the module. |

**The key rule:** Always attempt to recall *before* you look. This is the difference between active recall and passive review.

### Interleaving in leaps

| Feature | How to use it |
|---------|---------------|
| Multiple active topics | Keep 2–3 topics in progress simultaneously. Switch between them each session. |
| `SHARED/concepts/` | Cross-topic concept files appear in multiple topics naturally, reinforcing connections. |
| Mixed exercises | After completing multiple modules, solve exercises drawn from multiple topics in a single session. |

## 5. Weekly Study Schedule Template

This schedule is designed for ~1 hour/day and maximizes all three techniques: spacing, active recall, and interleaving.

```
Monday
  - (15 min) Review: Check Dataview Query 9 for due items. Recall from memory, then check.
  - (30 min) Study: New module in Topic A — read once, then close and write key points.
  - (15 min) Practice: Attempt 2–3 exercises from Topic A.

Tuesday
  - (15 min) Review: Due items from yesterday's Topic A module (set next_review = today + 1).
  - (30 min) Study: New module in Topic B (interleave with A).
  - (15 min) Questions: Write 5 questions about Topic B module from memory.

Wednesday
  - (15 min) Review: Due items (Topic A day-2 review + any others).
  - (30 min) Study: Continue Topic A — next module.
  - (15 min) Exercises: Topic B exercises.

Thursday
  - (15 min) Review: Due items.
  - (30 min) Study: Topic C (third active topic — maximize interleaving).
  - (15 min) Self-test: Take the test from the Topic A module completed Monday.

Friday
  - (15 min) Review: Due items.
  - (20 min) Mixed exercises: Pull one exercise from each of A, B, and C.
  - (25 min) Write: Explain in plain English the hardest concept from this week.

Weekend (30 min total)
  - Review any due items.
  - Update grades.md with self-test scores.
  - Plan next week's modules.
```

**Adjusting for time:**
- 30 min/day: Cut new study to 15 min, keep review full.
- 2 hours/day: Double the exercise time; add a second topic per session.

**The non-negotiable rule:** Do the review *first*, every session. It is the highest-leverage 15 minutes you will spend.

## 6. Summary: Key Principles

### The Science in Three Sentences

1. Memory decays exponentially without review, but each successful review increases both retention and the interval before the next review is needed.
2. Actively retrieving information (testing yourself) is 2–4× more effective than passively re-reading the same material, even when it feels harder.
3. Mixing topics and problem types in a single session (interleaving) produces better long-term retention than drilling one topic at a time, even though it feels less productive in the moment.

### The leaps Principles

| Principle | Implementation in leaps |
|-----------|-------------------------|
| Space your reviews | `next_review` frontmatter + Dataview Query 9 dashboard |
| Test before you read | `questions.md` + module test sections |
| Interleave topics | Keep multiple active topics; use `SHARED/concepts/` cross-links |
| Record everything | `grades.md` (append-only) — honest track record |
| Make it retrievable | Structured modules with consistent headings — searchable, linkable |
| Build in public | git history is your learning timeline — every commit is a study session |

---

### Further Reading

- Roediger & Karpicke (2006) — *Test-Enhanced Learning* — the foundational retrieval practice paper
- Cepeda et al. (2006) — *Distributed Practice in Verbal Recall Tasks* — meta-analysis of spacing
- Kornell & Bjork (2008) — *Learning Concepts and Categories* — interleaving research
- Wozniak (1990) — *Optimization of Learning* — the SuperMemo/SM-2 algorithm paper
- Brown, Roediger & McDaniel (2014) — *Make It Stick* — accessible book covering all of the above

---

*This notebook is part of the [leaps](https://github.com/your-org/leaps) repository. See the `notebooks/README.md` for other cross-topic notebooks.*